# SASRec Attention-Bias Refinement BPI2012 Colab Train (`refine_ml50_do035` baseline)

Refinement notebook for the most promising time-aware direction so far: attention bias.

This notebook keeps the same setting as the previous attention-bias experiment, except for one change:
- bucket boundaries are refined to make short time gaps more granular.

Comparison targets:
- baseline `refine_ml50_do035`
- previous attention-bias setting (`60,600,3600,86400,604800`)
- refined attention-bias setting (`10,60,600,3600,86400,604800`)


In [1]:
import torch

print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu name:', torch.cuda.get_device_name(0))


torch version: 2.10.0+cu128
cuda available: True
gpu name: Tesla T4


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
GITHUB_USERNAME = 'hwbuzz'

DRIVE_ROOT = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction'
REPO_DIR = '/content/time-aware-behavior-prediction'

DATA_DIR = f'{DRIVE_ROOT}/data/processed/bpi2012_complete_only'
BASELINE_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10'
BASELINE_NDCG5_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5'
BASE_ATTN_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg10'
BASE_ATTN_NDCG5_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg5'
REFINE_ATTN_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_refine_ndcg10'
REFINE_ATTN_NDCG5_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_refine_ndcg5'
NOTEBOOK_DIR = f'{DRIVE_ROOT}/notebooks'

print('DATA_DIR:', DATA_DIR)
print('BASELINE_NDCG10_OUTPUT_DIR:', BASELINE_NDCG10_OUTPUT_DIR)
print('BASELINE_NDCG5_OUTPUT_DIR:', BASELINE_NDCG5_OUTPUT_DIR)
print('BASE_ATTN_NDCG10_OUTPUT_DIR:', BASE_ATTN_NDCG10_OUTPUT_DIR)
print('BASE_ATTN_NDCG5_OUTPUT_DIR:', BASE_ATTN_NDCG5_OUTPUT_DIR)
print('REFINE_ATTN_NDCG10_OUTPUT_DIR:', REFINE_ATTN_NDCG10_OUTPUT_DIR)
print('REFINE_ATTN_NDCG5_OUTPUT_DIR:', REFINE_ATTN_NDCG5_OUTPUT_DIR)


DATA_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only
BASELINE_NDCG10_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10
BASELINE_NDCG5_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5
BASE_ATTN_NDCG10_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg10
BASE_ATTN_NDCG5_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg5
REFINE_ATTN_NDCG10_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_refine_ndcg10
REFINE_ATTN_NDCG5_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_refine_ndcg5


In [4]:
!mkdir -p "$NOTEBOOK_DIR"
!mkdir -p "$DATA_DIR"
!mkdir -p "$BASELINE_NDCG10_OUTPUT_DIR"
!mkdir -p "$BASELINE_NDCG5_OUTPUT_DIR"
!mkdir -p "$BASE_ATTN_NDCG10_OUTPUT_DIR"
!mkdir -p "$BASE_ATTN_NDCG5_OUTPUT_DIR"
!mkdir -p "$REFINE_ATTN_NDCG10_OUTPUT_DIR"
!mkdir -p "$REFINE_ATTN_NDCG5_OUTPUT_DIR"


In [5]:
%cd /content
!test -d time-aware-behavior-prediction || git clone https://github.com/$GITHUB_USERNAME/time-aware-behavior-prediction.git
%cd /content/time-aware-behavior-prediction
!git pull


/content
/content/time-aware-behavior-prediction
Already up to date.


In [6]:
%cd /content/time-aware-behavior-prediction

skip_packages = ['pywinpty']

with open('requirements.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

with open('requirements_colab.txt', 'w', encoding='utf-8') as f:
    for line in lines:
        pkg = line.strip().lower()
        if not any(name in pkg for name in skip_packages):
            f.write(line)

print('created requirements_colab.txt')


/content/time-aware-behavior-prediction
created requirements_colab.txt


In [7]:
!pip install -r requirements_colab.txt


In [8]:
!ls "$DATA_DIR"


events_complete_only_filtered.csv  sasrec_interactions.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   user_map.csv


In [9]:
%cd /content/time-aware-behavior-prediction
!mkdir -p data/processed
!cp -r "$DATA_DIR" data/processed/
!ls data/processed/bpi2012_complete_only


/content/time-aware-behavior-prediction
events_complete_only_filtered.csv  sasrec_interactions.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   user_map.csv


## Experiment design

Fixed baseline setting:
- `refine_ml50_do035`
- `hidden_units=50, num_blocks=2, num_heads=1, maxlen=50, lr=0.001, dropout=0.35`
- seeds: `42`, `2024`, `7`

Attention-bias design:
- `delta_start_seconds`
- causal pairwise gap
- refined boundaries: `10,60,600,3600,86400,604800`
- objective: check whether finer short-term buckets improve over the original attention-bias setting


## Check prerequisite runs


In [10]:
from pathlib import Path

checks = [
    ('Baseline NDCG@10', Path(BASELINE_NDCG10_OUTPUT_DIR), [
        'refine_ml50_do035_s42',
        'refine_ml50_do035_s2024',
        'refine_ml50_do035_s7',
    ]),
    ('Baseline NDCG@5', Path(BASELINE_NDCG5_OUTPUT_DIR), [
        'refine_ml50_do035_s42',
        'refine_ml50_do035_s2024',
        'refine_ml50_do035_s7',
    ]),
    ('Base attention-bias NDCG@10', Path(BASE_ATTN_NDCG10_OUTPUT_DIR), [
        'attnbias_dstart_ml50_do035_b9_s42',
        'attnbias_dstart_ml50_do035_b9_s2024',
        'attnbias_dstart_ml50_do035_b9_s7',
    ]),
    ('Base attention-bias NDCG@5', Path(BASE_ATTN_NDCG5_OUTPUT_DIR), [
        'attnbias_dstart_ml50_do035_b9_s42',
        'attnbias_dstart_ml50_do035_b9_s2024',
        'attnbias_dstart_ml50_do035_b9_s7',
    ]),
]

for label, output_dir, run_names in checks:
    print('=' * 80)
    print(label)
    for run_name in run_names:
        run_dir = output_dir / run_name
        print(run_name, 'EXISTS' if run_dir.exists() else 'MISSING')


Baseline NDCG@10
refine_ml50_do035_s42 EXISTS
refine_ml50_do035_s2024 EXISTS
refine_ml50_do035_s7 EXISTS
Baseline NDCG@5
refine_ml50_do035_s42 EXISTS
refine_ml50_do035_s2024 EXISTS
refine_ml50_do035_s7 EXISTS
Base attention-bias NDCG@10
attnbias_dstart_ml50_do035_b9_s42 EXISTS
attnbias_dstart_ml50_do035_b9_s2024 EXISTS
attnbias_dstart_ml50_do035_b9_s7 EXISTS
Base attention-bias NDCG@5
attnbias_dstart_ml50_do035_b9_s42 EXISTS
attnbias_dstart_ml50_do035_b9_s2024 EXISTS
attnbias_dstart_ml50_do035_b9_s7 EXISTS


## Check planned refinement runs


In [11]:
planned_ndcg10 = [
    'attnbias_dstart_ml50_do035_b10_s42',
    'attnbias_dstart_ml50_do035_b10_s2024',
    'attnbias_dstart_ml50_do035_b10_s7',
]
planned_ndcg5 = [
    'attnbias_dstart_ml50_do035_b10_s42',
    'attnbias_dstart_ml50_do035_b10_s2024',
    'attnbias_dstart_ml50_do035_b10_s7',
]

for label, output_dir, run_names in [
    ('Refined attention-bias NDCG@10', Path(REFINE_ATTN_NDCG10_OUTPUT_DIR), planned_ndcg10),
    ('Refined attention-bias NDCG@5', Path(REFINE_ATTN_NDCG5_OUTPUT_DIR), planned_ndcg5),
]:
    print('=' * 80)
    print(label)
    for run_name in run_names:
        run_dir = output_dir / run_name
        print(run_name, 'EXISTS' if run_dir.exists() else 'OK')


Refined attention-bias NDCG@10
attnbias_dstart_ml50_do035_b10_s42 OK
attnbias_dstart_ml50_do035_b10_s2024 OK
attnbias_dstart_ml50_do035_b10_s7 OK
Refined attention-bias NDCG@5
attnbias_dstart_ml50_do035_b10_s42 OK
attnbias_dstart_ml50_do035_b10_s2024 OK
attnbias_dstart_ml50_do035_b10_s7 OK


## Train refined attention-bias runs for `NDCG@10`


### attnbias_dstart_ml50_do035_b10_s42


In [12]:
!python src/train_sasrec.py \
  --run_name attnbias_dstart_ml50_do035_b10_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 42 \
  --use_time_attention_bias \
  --time_delta_column delta_start_seconds \
  --time_bucket_boundaries 10,60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_refine_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_refine_ndcg10/attnbias_dstart_ml50_do035_b10_s42
epoch=1, loss=0.5885
epoch=2, loss=0.2657
epoch=3, loss=0.2038
epoch=4, loss=0.1698
epoch=5, loss=0.1461
valid [full], NDCG@5: 0.6390, HR@5: 0.7473, NDCG@10: 0.6837, HR@10: 0.8904, MRR: 0.6315
valid [sampled], NDCG@5: 0.5562, HR@5: 0.5570, NDCG@10: 0.5634, HR@10: 0.5806, MRR: 0.5720
test [full], NDCG@5: 0.5939, HR@5: 0.7428, NDCG@10: 0.6336, HR@10: 0.8731, MRR: 0.5682
test [sampled], NDCG@5: 0.2043, HR@5: 0.2093, NDCG@10: 0.2483, HR@10: 0.3516, MRR: 0.2488
saved eval checkpoint: /content/drive/MyDrive/ai-pro

### attnbias_dstart_ml50_do035_b10_s2024


In [13]:
!python src/train_sasrec.py \
  --run_name attnbias_dstart_ml50_do035_b10_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 2024 \
  --use_time_attention_bias \
  --time_delta_column delta_start_seconds \
  --time_bucket_boundaries 10,60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_refine_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_refine_ndcg10/attnbias_dstart_ml50_do035_b10_s2024
epoch=1, loss=0.6100
epoch=2, loss=0.2722
epoch=3, loss=0.1996
epoch=4, loss=0.1620
epoch=5, loss=0.1431
valid [full], NDCG@5: 0.6673, HR@5: 0.8155, NDCG@10: 0.7273, HR@10: 0.9900, MRR: 0.6480
valid [sampled], NDCG@5: 0.5491, HR@5: 0.5583, NDCG@10: 0.5606, HR@10: 0.5939, MRR: 0.5652
test [full], NDCG@5: 0.7594, HR@5: 0.9086, NDCG@10: 0.7896, HR@10: 1.0000, MRR: 0.7240
test [sampled], NDCG@5: 0.2071, HR@5: 0.2694, NDCG@10: 0.2644, HR@10: 0.4459, MRR: 0.2391
saved eval checkpoint: /content/drive/MyDrive/ai-p

### attnbias_dstart_ml50_do035_b10_s7


In [14]:
!python src/train_sasrec.py \
  --run_name attnbias_dstart_ml50_do035_b10_s7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 7 \
  --use_time_attention_bias \
  --time_delta_column delta_start_seconds \
  --time_bucket_boundaries 10,60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_refine_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_refine_ndcg10/attnbias_dstart_ml50_do035_b10_s7
epoch=1, loss=0.5907
epoch=2, loss=0.2821
epoch=3, loss=0.2070
epoch=4, loss=0.1717
epoch=5, loss=0.1501
valid [full], NDCG@5: 0.6362, HR@5: 0.7450, NDCG@10: 0.7027, HR@10: 0.9483, MRR: 0.6341
valid [sampled], NDCG@5: 0.5579, HR@5: 0.5587, NDCG@10: 0.5621, HR@10: 0.5722, MRR: 0.5735
test [full], NDCG@5: 0.6338, HR@5: 0.7886, NDCG@10: 0.6792, HR@10: 0.9297, MRR: 0.6052
test [sampled], NDCG@5: 0.2451, HR@5: 0.2508, NDCG@10: 0.2872, HR@10: 0.3868, MRR: 0.2892
saved eval checkpoint: /content/drive/MyDrive/ai-proj

## Train refined attention-bias runs for `NDCG@5`


### attnbias_dstart_ml50_do035_b10_s42


In [15]:
!python src/train_sasrec.py \
  --run_name attnbias_dstart_ml50_do035_b10_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 42 \
  --use_time_attention_bias \
  --time_delta_column delta_start_seconds \
  --time_bucket_boundaries 10,60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_refine_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_refine_ndcg5/attnbias_dstart_ml50_do035_b10_s42
epoch=1, loss=0.5885
epoch=2, loss=0.2657
epoch=3, loss=0.2038
epoch=4, loss=0.1698
epoch=5, loss=0.1461
valid [full], NDCG@5: 0.6390, HR@5: 0.7473, NDCG@10: 0.6837, HR@10: 0.8904, MRR: 0.6315
valid [sampled], NDCG@5: 0.5562, HR@5: 0.5570, NDCG@10: 0.5634, HR@10: 0.5806, MRR: 0.5720
test [full], NDCG@5: 0.5939, HR@5: 0.7428, NDCG@10: 0.6336, HR@10: 0.8731, MRR: 0.5682
test [sampled], NDCG@5: 0.2043, HR@5: 0.2093, NDCG@10: 0.2483, HR@10: 0.3516, MRR: 0.2488
saved eval checkpoint: /content/drive/MyDrive/ai-proje

### attnbias_dstart_ml50_do035_b10_s2024


In [16]:
!python src/train_sasrec.py \
  --run_name attnbias_dstart_ml50_do035_b10_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 2024 \
  --use_time_attention_bias \
  --time_delta_column delta_start_seconds \
  --time_bucket_boundaries 10,60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_refine_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_refine_ndcg5/attnbias_dstart_ml50_do035_b10_s2024
epoch=1, loss=0.6100
epoch=2, loss=0.2722
epoch=3, loss=0.1996
epoch=4, loss=0.1620
epoch=5, loss=0.1431
valid [full], NDCG@5: 0.6673, HR@5: 0.8155, NDCG@10: 0.7273, HR@10: 0.9900, MRR: 0.6480
valid [sampled], NDCG@5: 0.5491, HR@5: 0.5583, NDCG@10: 0.5606, HR@10: 0.5939, MRR: 0.5652
test [full], NDCG@5: 0.7594, HR@5: 0.9086, NDCG@10: 0.7896, HR@10: 1.0000, MRR: 0.7240
test [sampled], NDCG@5: 0.2071, HR@5: 0.2694, NDCG@10: 0.2644, HR@10: 0.4459, MRR: 0.2391
saved eval checkpoint: /content/drive/MyDrive/ai-pro

### attnbias_dstart_ml50_do035_b10_s7


In [17]:
!python src/train_sasrec.py \
  --run_name attnbias_dstart_ml50_do035_b10_s7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 7 \
  --use_time_attention_bias \
  --time_delta_column delta_start_seconds \
  --time_bucket_boundaries 10,60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_refine_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_refine_ndcg5/attnbias_dstart_ml50_do035_b10_s7
epoch=1, loss=0.5907
epoch=2, loss=0.2821
epoch=3, loss=0.2070
epoch=4, loss=0.1717
epoch=5, loss=0.1501
valid [full], NDCG@5: 0.6362, HR@5: 0.7450, NDCG@10: 0.7027, HR@10: 0.9483, MRR: 0.6341
valid [sampled], NDCG@5: 0.5579, HR@5: 0.5587, NDCG@10: 0.5621, HR@10: 0.5722, MRR: 0.5735
test [full], NDCG@5: 0.6338, HR@5: 0.7886, NDCG@10: 0.6792, HR@10: 0.9297, MRR: 0.6052
test [sampled], NDCG@5: 0.2451, HR@5: 0.2508, NDCG@10: 0.2872, HR@10: 0.3868, MRR: 0.2892
saved eval checkpoint: /content/drive/MyDrive/ai-projec

## Rebuild result tables


In [18]:
from pathlib import Path
import json
import pandas as pd

def rebuild_df(output_dir: str):
    rows = []
    output_path = Path(output_dir)
    if not output_path.exists():
        return pd.DataFrame()
    for run_dir in output_path.iterdir():
        if not run_dir.is_dir():
            continue
        summary_path = run_dir / 'metrics_summary.json'
        config_path = run_dir / 'config.json'
        if not summary_path.exists() or not config_path.exists():
            continue
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        config = json.loads(config_path.read_text(encoding='utf-8'))
        row = {
            'run_name': summary.get('run_name'),
            'run_dir': str(run_dir),
            'completed_at': summary.get('completed_at'),
            'best_epoch': summary.get('best_epoch'),
            'checkpoint_best': summary.get('checkpoint_best'),
            'checkpoint_last': summary.get('checkpoint_last'),
            'metrics_history': summary.get('metrics_history'),
            'config_path': str(config_path),
            'metrics_summary': str(summary_path),
            'maxlen': config.get('maxlen'),
            'dropout_rate': config.get('dropout_rate'),
            'hidden_units': config.get('hidden_units'),
            'seed': config.get('seed'),
            'selection_metric': config.get('selection_metric'),
            'use_time_embedding': config.get('use_time_embedding', False),
            'use_time_attention_bias': config.get('use_time_attention_bias', False),
            'time_modeling_mode': config.get('time_modeling_mode'),
            'time_encoding': config.get('time_encoding'),
            'time_delta_column': config.get('time_delta_column'),
            'time_bucket_boundaries_parsed': config.get('time_bucket_boundaries_parsed'),
            'time_attention_bias_bucket_count': config.get('time_attention_bias_bucket_count'),
        }
        best_valid = summary.get('best_valid', {})
        best_test = summary.get('best_test_at_best_valid', {})
        def pick(metrics_group, mode, key):
            return metrics_group.get(mode, {}).get(key)
        row.update({
            'best_valid_full_ndcg@10': pick(best_valid, 'full', 'ndcg@10'),
            'best_valid_full_hr@10': pick(best_valid, 'full', 'hr@10'),
            'best_valid_full_ndcg@5': pick(best_valid, 'full', 'ndcg@5'),
            'best_valid_full_hr@5': pick(best_valid, 'full', 'hr@5'),
            'best_valid_full_mrr': pick(best_valid, 'full', 'mrr'),
            'best_test_full_ndcg@10': pick(best_test, 'full', 'ndcg@10'),
            'best_test_full_hr@10': pick(best_test, 'full', 'hr@10'),
            'best_test_full_ndcg@5': pick(best_test, 'full', 'ndcg@5'),
            'best_test_full_hr@5': pick(best_test, 'full', 'hr@5'),
            'best_test_full_mrr': pick(best_test, 'full', 'mrr'),
            'best_valid_sampled_ndcg@10': pick(best_valid, 'sampled', 'ndcg@10'),
            'best_valid_sampled_hr@10': pick(best_valid, 'sampled', 'hr@10'),
            'best_valid_sampled_ndcg@5': pick(best_valid, 'sampled', 'ndcg@5'),
            'best_valid_sampled_hr@5': pick(best_valid, 'sampled', 'hr@5'),
            'best_valid_sampled_mrr': pick(best_valid, 'sampled', 'mrr'),
            'best_test_sampled_ndcg@10': pick(best_test, 'sampled', 'ndcg@10'),
            'best_test_sampled_hr@10': pick(best_test, 'sampled', 'hr@10'),
            'best_test_sampled_ndcg@5': pick(best_test, 'sampled', 'ndcg@5'),
            'best_test_sampled_hr@5': pick(best_test, 'sampled', 'hr@5'),
            'best_test_sampled_mrr': pick(best_test, 'sampled', 'mrr'),
        })
        rows.append(row)
    return pd.DataFrame(rows)


In [19]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.max_colwidth", None)

## NDCG@10 comparison summary


In [20]:
baseline_runs = [
    'refine_ml50_do035_s42',
    'refine_ml50_do035_s2024',
    'refine_ml50_do035_s7',
]
base_attn_runs = [
    'attnbias_dstart_ml50_do035_b9_s42',
    'attnbias_dstart_ml50_do035_b9_s2024',
    'attnbias_dstart_ml50_do035_b9_s7',
]
refine_attn_runs = [
    'attnbias_dstart_ml50_do035_b10_s42',
    'attnbias_dstart_ml50_do035_b10_s2024',
    'attnbias_dstart_ml50_do035_b10_s7',
]

baseline_df = rebuild_df(BASELINE_NDCG10_OUTPUT_DIR)
base_attn_df = rebuild_df(BASE_ATTN_NDCG10_OUTPUT_DIR)
refine_attn_df = rebuild_df(REFINE_ATTN_NDCG10_OUTPUT_DIR)

baseline_subset = baseline_df[baseline_df['run_name'].isin(baseline_runs)].copy()
baseline_subset['time_variant'] = 'baseline'

base_attn_subset = base_attn_df[base_attn_df['run_name'].isin(base_attn_runs)].copy()
base_attn_subset['time_variant'] = 'attention_bias_b9_base'

refine_attn_subset = refine_attn_df[refine_attn_df['run_name'].isin(refine_attn_runs)].copy()
refine_attn_subset['time_variant'] = 'attention_bias_b10'

df_ndcg10 = pd.concat([baseline_subset, base_attn_subset, refine_attn_subset], ignore_index=True)
df_ndcg10 = df_ndcg10.sort_values(['time_variant', 'seed', 'run_name']).reset_index(drop=True)
df_ndcg10[[
    'run_name', 'seed', 'time_variant', 'time_delta_column', 'time_bucket_boundaries_parsed', 'time_attention_bias_bucket_count',
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10',
    'best_valid_full_ndcg@5', 'best_valid_full_hr@5',
    'best_valid_full_mrr',
    'best_test_full_ndcg@10', 'best_test_full_hr@10',
    'best_test_full_ndcg@5', 'best_test_full_hr@5',
    'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10',
    'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5',
    'best_valid_sampled_mrr',
    'best_test_sampled_ndcg@10', 'best_test_sampled_hr@10',
    'best_test_sampled_ndcg@5', 'best_test_sampled_hr@5',
    'best_test_sampled_mrr',
]]


,run_name,seed,time_variant,time_delta_column,time_bucket_boundaries_parsed,time_attention_bias_bucket_count,best_valid_full_ndcg@10,best_valid_full_hr@10,best_valid_full_ndcg@5,best_valid_full_hr@5,best_valid_full_mrr,best_test_full_ndcg@10,best_test_full_hr@10,best_test_full_ndcg@5,best_test_full_hr@5,best_test_full_mrr,best_valid_sampled_ndcg@10,best_valid_sampled_hr@10,best_valid_sampled_ndcg@5,best_valid_sampled_hr@5,best_valid_sampled_mrr,best_test_sampled_ndcg@10,best_test_sampled_hr@10,best_test_sampled_ndcg@5,best_test_sampled_hr@5,best_test_sampled_mrr
0,attnbias_dstart_ml50_do035_b10_s7,7,attention_bias_b10,delta_start_seconds,"[10.0, 60.0, 600.0, 3600.0, 86400.0, 604800.0]",8,0.767971,0.974858,0.732792,0.860368,0.706519,0.848382,1.000000,0.848158,0.999323,0.795808,0.633300,0.696467,0.610889,0.626087,0.627294,0.461120,0.625440,0.393811,0.411000,0.438855
1,attnbias_dstart_ml50_do035_b10_s42,42,attention_bias_b10,delta_start_seconds,"[10.0, 60.0, 600.0, 3600.0, 86400.0, 604800.0]",8,0.738445,0.970552,0.706377,0.867010,0.668807,0.753725,1.000000,0.736480,0.947262,0.671027,0.588010,0.641400,0.568729,0.580867,0.587662,0.220428,0.460106,0.141005,0.214082,0.179538
2,attnbias_dstart_ml50_do035_b10_s2024,2024,attention_bias_b10,delta_start_seconds,"[10.0, 60.0, 600.0, 3600.0, 86400.0, 604800.0]",8,0.730977,0.945803,0.701896,0.851263,0.668709,0.896548,0.999865,0.877658,0.941724,0.863958,0.574793,0.618808,0.558018,0.565732,0.577309,0.415633,0.605827,0.351656,0.407182,0.381936
3,attnbias_dstart_ml50_do035_b9_s7,7,attention_bias_b9_base,delta_start_seconds,"[60.0, 600.0, 3600.0, 86400.0, 604800.0]",7,0.755366,0.953861,0.714870,0.824671,0.699184,0.846762,1.000000,0.846665,0.999728,0.793978,0.623265,0.666077,0.607399,0.615479,0.623619,0.497216,0.628223,0.442098,0.450204,0.485290
4,attnbias_dstart_ml50_do035_b9_s42,42,attention_bias_b9_base,delta_start_seconds,"[60.0, 600.0, 3600.0, 86400.0, 604800.0]",7,0.741856,0.969750,0.709565,0.864487,0.673783,0.785724,0.999865,0.764719,0.935724,0.714860,0.585005,0.635957,0.566501,0.577970,0.585189,0.199334,0.453522,0.115783,0.193920,0.156593
5,attnbias_dstart_ml50_do035_b9_s2024,2024,attention_bias_b9_base,delta_start_seconds,"[60.0, 600.0, 3600.0, 86400.0, 604800.0]",7,0.735385,0.971475,0.702964,0.865254,0.665031,0.885738,0.999865,0.863911,0.928234,0.850981,0.566216,0.612548,0.548977,0.558247,0.568592,0.428086,0.626287,0.366083,0.436179,0.388188
6,refine_ml50_do035_s7,7,baseline,delta_prev_seconds,None,None,0.728826,0.961737,0.696223,0.857259,0.660451,0.872416,1.000000,0.859037,0.957678,0.831208,0.577254,0.611043,0.565802,0.575315,0.581977,0.403018,0.656933,0.324827,0.417493,0.345750
7,refine_ml50_do035_s42,42,baseline,delta_prev_seconds,None,None,0.735378,0.977276,0.697410,0.854186,0.663510,0.891774,1.000000,0.868726,0.926375,0.858375,0.568693,0.606441,0.555385,0.565093,0.572753,0.456024,0.640631,0.399455,0.467411,0.419230
8,refine_ml50_do035_s2024,2024,baseline,delta_prev_seconds,None,None,0.741573,0.992973,0.713154,0.906351,0.664353,0.775861,1.000000,0.727453,0.850196,0.707187,0.572972,0.597043,0.563603,0.567069,0.582973,0.257972,0.431769,0.203548,0.264514,0.234183


In [21]:
summary_ndcg10 = df_ndcg10.groupby('time_variant')[[
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10',
    'best_valid_full_ndcg@5', 'best_valid_full_hr@5',
    'best_valid_full_mrr',
    'best_test_full_ndcg@10', 'best_test_full_hr@10',
    'best_test_full_ndcg@5', 'best_test_full_hr@5',
    'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10',
    'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5',
    'best_valid_sampled_mrr',
    'best_test_sampled_ndcg@10', 'best_test_sampled_hr@10',
    'best_test_sampled_ndcg@5', 'best_test_sampled_hr@5',
    'best_test_sampled_mrr',
]].agg(['mean', 'std'])
summary_ndcg10


best_valid_full_ndcg@10           best_valid_full_hr@10           best_valid_full_ndcg@5           best_valid_full_hr@5           best_valid_full_mrr           best_test_full_ndcg@10           best_test_full_hr@10           best_test_full_ndcg@5           best_test_full_hr@5           best_test_full_mrr           best_valid_sampled_ndcg@10           best_valid_sampled_hr@10           best_valid_sampled_ndcg@5           best_valid_sampled_hr@5           best_valid_sampled_mrr           best_test_sampled_ndcg@10           best_test_sampled_hr@10           best_test_sampled_ndcg@5           best_test_sampled_hr@5           best_test_sampled_mrr          
                                          mean       std                  mean       std                   mean       std                 mean       std                mean       std                   mean       std                 mean       std                  mean       std                mean       std               mean       std                       mean       std                     mean       std                      mean       std                    mean       std                   mean       std                      mean       std                    mean       std                     mean       std                   mean       std                  mean       std
time_variant                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              
attention_bias_b10                    0.745798  0.019562              0.963738  0.015681               0.713688  0.016695             0.859547  0.007906            0.681345  0.021801               0.832885  0.072662             0.999955  0.000078              0.820765  0.074469            0.962770  0.031777           0.776931  0.097841                   0.598701  0.030684                 0.652225  0.039945                  0.579212  0.027951                0.590896  0.031402               0.597422  0.026383                  0.365727  0.127871                0.563791  0.090328                 0.295491  0.135439               0.344088  0.112605              0.333443  0.136290
attention_bias_b9_base                0.744203  0.010195              0.965029  0.009710               0.709133  0.005965             0.851471  0.023212            0.679333  0.017740               0.839408  0.050411             0.999910  0.000078              0.825099  0.052996            0.954562  0.039294           0.786607  0.068359                   0.591495  0.029073                 0.638194  0.026835                  0.574292  0.029980                0.583899  0.029073               0.592467  0.028226                  0.374879  0.155906                0.569344  0.100310                 0.307988  0.170739               0.360101  0.144087              0.343357  0.168872
baseline                              0.735259  0.006374              0.977329  0.015618               0.702263  0.009451             0.872599  0.029271            0.662771  0.002053               0.846684  0.062093             1.000000  0.000000              0.818405  0.078916            0.911416  0.055280           0.798923  0.080599                   0.572973  0.004280                 0.604843  0.007136                  0.561597  0.005491                0.569159  0.005422               0.579235  0.005635                  0.372338  0.102528                0.576445  0.125558                 0.309277 

Interpretation guide for NDCG@10:
- compare `best_valid_full_ndcg@10` and `best_test_full_ndcg@10` first
- then check whether the refined attention-bias boundaries improve over the original attention-bias run
- sampled and MRR are supplementary


## NDCG@5 comparison summary


In [22]:
baseline_runs = [
    'refine_ml50_do035_s42',
    'refine_ml50_do035_s2024',
    'refine_ml50_do035_s7',
]
base_attn_runs = [
    'attnbias_dstart_ml50_do035_b9_s42',
    'attnbias_dstart_ml50_do035_b9_s2024',
    'attnbias_dstart_ml50_do035_b9_s7',
]
refine_attn_runs = [
    'attnbias_dstart_ml50_do035_b10_s42',
    'attnbias_dstart_ml50_do035_b10_s2024',
    'attnbias_dstart_ml50_do035_b10_s7',
]

baseline_df = rebuild_df(BASELINE_NDCG5_OUTPUT_DIR)
base_attn_df = rebuild_df(BASE_ATTN_NDCG5_OUTPUT_DIR)
refine_attn_df = rebuild_df(REFINE_ATTN_NDCG5_OUTPUT_DIR)

baseline_subset = baseline_df[baseline_df['run_name'].isin(baseline_runs)].copy()
baseline_subset['time_variant'] = 'baseline'

base_attn_subset = base_attn_df[base_attn_df['run_name'].isin(base_attn_runs)].copy()
base_attn_subset['time_variant'] = 'attention_bias_b9_base'

refine_attn_subset = refine_attn_df[refine_attn_df['run_name'].isin(refine_attn_runs)].copy()
refine_attn_subset['time_variant'] = 'attention_bias_b10'

df_ndcg5 = pd.concat([baseline_subset, base_attn_subset, refine_attn_subset], ignore_index=True)
df_ndcg5 = df_ndcg5.sort_values(['time_variant', 'seed', 'run_name']).reset_index(drop=True)
df_ndcg5[[
    'run_name', 'seed', 'time_variant', 'time_delta_column', 'time_bucket_boundaries_parsed', 'time_attention_bias_bucket_count',
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10',
    'best_valid_full_ndcg@5', 'best_valid_full_hr@5',
    'best_valid_full_mrr',
    'best_test_full_ndcg@10', 'best_test_full_hr@10',
    'best_test_full_ndcg@5', 'best_test_full_hr@5',
    'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10',
    'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5',
    'best_valid_sampled_mrr',
    'best_test_sampled_ndcg@10', 'best_test_sampled_hr@10',
    'best_test_sampled_ndcg@5', 'best_test_sampled_hr@5',
    'best_test_sampled_mrr',
]]


,run_name,seed,time_variant,time_delta_column,time_bucket_boundaries_parsed,time_attention_bias_bucket_count,best_valid_full_ndcg@10,best_valid_full_hr@10,best_valid_full_ndcg@5,best_valid_full_hr@5,best_valid_full_mrr,best_test_full_ndcg@10,best_test_full_hr@10,best_test_full_ndcg@5,best_test_full_hr@5,best_test_full_mrr,best_valid_sampled_ndcg@10,best_valid_sampled_hr@10,best_valid_sampled_ndcg@5,best_valid_sampled_hr@5,best_valid_sampled_mrr,best_test_sampled_ndcg@10,best_test_sampled_hr@10,best_test_sampled_ndcg@5,best_test_sampled_hr@5,best_test_sampled_mrr
0,attnbias_dstart_ml50_do035_b10_s7,7,attention_bias_b10,delta_start_seconds,"[10.0, 60.0, 600.0, 3600.0, 86400.0, 604800.0]",8,0.767971,0.974858,0.732792,0.860368,0.706519,0.848382,1.000000,0.848158,0.999323,0.795808,0.633300,0.696467,0.610889,0.626087,0.627294,0.461120,0.625440,0.393811,0.411000,0.438855
1,attnbias_dstart_ml50_do035_b10_s42,42,attention_bias_b10,delta_start_seconds,"[10.0, 60.0, 600.0, 3600.0, 86400.0, 604800.0]",8,0.738445,0.970552,0.706377,0.867010,0.668807,0.753725,1.000000,0.736480,0.947262,0.671027,0.588010,0.641400,0.568729,0.580867,0.587662,0.220428,0.460106,0.141005,0.214082,0.179538
2,attnbias_dstart_ml50_do035_b10_s2024,2024,attention_bias_b10,delta_start_seconds,"[10.0, 60.0, 600.0, 3600.0, 86400.0, 604800.0]",8,0.730977,0.945803,0.701896,0.851263,0.668709,0.896548,0.999865,0.877658,0.941724,0.863958,0.574793,0.618808,0.558018,0.565732,0.577309,0.415633,0.605827,0.351656,0.407182,0.381936
3,attnbias_dstart_ml50_do035_b9_s7,7,attention_bias_b9_base,delta_start_seconds,"[60.0, 600.0, 3600.0, 86400.0, 604800.0]",7,0.765708,0.975845,0.728865,0.854661,0.703877,0.847794,1.000000,0.847697,0.999728,0.795264,0.625614,0.676007,0.606813,0.616159,0.624051,0.500175,0.632429,0.444546,0.452782,0.487732
4,attnbias_dstart_ml50_do035_b9_s42,42,attention_bias_b9_base,delta_start_seconds,"[60.0, 600.0, 3600.0, 86400.0, 604800.0]",7,0.741824,0.969750,0.709060,0.862317,0.673822,0.783187,0.999865,0.763183,0.935995,0.711912,0.585252,0.636770,0.566501,0.577970,0.585234,0.194366,0.445244,0.112266,0.190256,0.152804
5,attnbias_dstart_ml50_do035_b9_s2024,2024,attention_bias_b9_base,delta_start_seconds,"[60.0, 600.0, 3600.0, 86400.0, 604800.0]",7,0.741407,0.975616,0.712340,0.881719,0.670798,0.894503,0.999865,0.878950,0.952884,0.861072,0.572542,0.619131,0.554208,0.560547,0.576476,0.456145,0.618822,0.406179,0.464726,0.428497
6,refine_ml50_do035_s7,7,baseline,delta_prev_seconds,None,None,0.728826,0.961737,0.696223,0.857259,0.660451,0.872416,1.000000,0.859037,0.957678,0.831208,0.577254,0.611043,0.565802,0.575315,0.581977,0.403018,0.656933,0.324827,0.417493,0.345750
7,refine_ml50_do035_s42,42,baseline,delta_prev_seconds,None,None,0.735378,0.977276,0.697410,0.854186,0.663510,0.891774,1.000000,0.868726,0.926375,0.858375,0.568693,0.606441,0.555385,0.565093,0.572753,0.456024,0.640631,0.399455,0.467411,0.419230
8,refine_ml50_do035_s2024,2024,baseline,delta_prev_seconds,None,None,0.741573,0.992973,0.713154,0.906351,0.664353,0.775861,1.000000,0.727453,0.850196,0.707187,0.572972,0.597043,0.563603,0.567069,0.582973,0.257972,0.431769,0.203548,0.264514,0.234183


In [23]:
summary_ndcg5 = df_ndcg5.groupby('time_variant')[[
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10',
    'best_valid_full_ndcg@5', 'best_valid_full_hr@5',
    'best_valid_full_mrr',
    'best_test_full_ndcg@10', 'best_test_full_hr@10',
    'best_test_full_ndcg@5', 'best_test_full_hr@5',
    'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10',
    'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5',
    'best_valid_sampled_mrr',
    'best_test_sampled_ndcg@10', 'best_test_sampled_hr@10',
    'best_test_sampled_ndcg@5', 'best_test_sampled_hr@5',
    'best_test_sampled_mrr',
]].agg(['mean', 'std'])
summary_ndcg5


best_valid_full_ndcg@10           best_valid_full_hr@10           best_valid_full_ndcg@5           best_valid_full_hr@5           best_valid_full_mrr           best_test_full_ndcg@10           best_test_full_hr@10           best_test_full_ndcg@5           best_test_full_hr@5           best_test_full_mrr           best_valid_sampled_ndcg@10           best_valid_sampled_hr@10           best_valid_sampled_ndcg@5           best_valid_sampled_hr@5           best_valid_sampled_mrr           best_test_sampled_ndcg@10           best_test_sampled_hr@10           best_test_sampled_ndcg@5           best_test_sampled_hr@5           best_test_sampled_mrr          
                                          mean       std                  mean       std                   mean       std                 mean       std                mean       std                   mean       std                 mean       std                  mean       std                mean       std               mean       std                       mean       std                     mean       std                      mean       std                    mean       std                   mean       std                      mean       std                    mean       std                     mean       std                   mean       std                  mean       std
time_variant                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              
attention_bias_b10                    0.745798  0.019562              0.963738  0.015681               0.713688  0.016695             0.859547  0.007906            0.681345  0.021801               0.832885  0.072662             0.999955  0.000078              0.820765  0.074469            0.962770  0.031777           0.776931  0.097841                   0.598701  0.030684                 0.652225  0.039945                  0.579212  0.027951                0.590896  0.031402               0.597422  0.026383                  0.365727  0.127871                0.563791  0.090328                 0.295491  0.135439               0.344088  0.112605              0.333443  0.136290
attention_bias_b9_base                0.749646  0.013911              0.973737  0.003455               0.716755  0.010615             0.866232  0.013947            0.682833  0.018287               0.841828  0.055897             0.999910  0.000078              0.829944  0.059891            0.962869  0.033019           0.789416  0.074752                   0.594469  0.027711                 0.643969  0.029113                  0.575841  0.027518                0.584892  0.028445               0.595254  0.025321                  0.383562  0.165321                0.565498  0.104365                 0.320997  0.181781               0.369255  0.155132              0.356344  0.178742
baseline                              0.735259  0.006374              0.977329  0.015618               0.702263  0.009451             0.872599  0.029271            0.662771  0.002053               0.846684  0.062093             1.000000  0.000000              0.818405  0.078916            0.911416  0.055280           0.798923  0.080599                   0.572973  0.004280                 0.604843  0.007136                  0.561597  0.005491                0.569159  0.005422               0.579235  0.005635                  0.372338  0.102528                0.576445  0.125558                 0.309277 

Interpretation guide for NDCG@5:
- compare `best_valid_full_ndcg@5` and `best_test_full_ndcg@5` first
- then check whether the refined attention-bias boundaries improve over the original attention-bias run
- sampled and MRR are supplementary
